In [1]:
import napari
import numpy as np
from aicsimageio import AICSImage

import pandas as pd
from skimage import measure


import tifffile as tf
import matplotlib.pyplot as plt
import xarray as xr
from tifffile import tifffile
import tifftools
import os
from matplotlib import pyplot as plt
from os.path import sep
from skimage import io
from PIL import Image
import imageio


from skimage.measure import regionprops_table

import seaborn as sns

import czifile
from czifile import CziFile

import math
import matplotlib.colors as mcolors

/Users/fisherguest/miniconda3/envs/sansachen_czi/lib/python3.11/site-packages/pydantic/_migration.py:283: UserWarning: `pydantic.error_wrappers:ValidationError` has been moved to `pydantic:ValidationError`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')


In [2]:
# this shows how many delta7 are there in each subtype

delta7_naming = pd.read_csv('neuprint_delta7_names.csv')
delta7_naming = delta7_naming.drop(columns=["notes"])
delta7_naming['instance'] = delta7_naming['instance'].str.replace(r'^Delta7\(PB15\)_', '', regex=True)
delta7_naming.sort_values('instance', inplace=True)
delta7_naming.reset_index(drop=True, inplace=True)
subtype_summary = delta7_naming.groupby("instance").size().reset_index(name="count")
subtype_summary

,instance,count
0,L1L9R8_R,5
1,L2R7_R,5
2,L3R6_R,4
3,L4R5_R,5
4,L4R6_R,2
5,L5R4_L,5
6,L6R3_L,5
7,L6R4_L,2
8,L7R2_L,3
9,L7R3_L,1


In [3]:
fly1 = '/Users/fisherguest/Documents/sansa images/08142025/SC-0814-fly1.czi'
fly2 = '/Users/fisherguest/Documents/sansa images/08142025/SC-0814-fly2.czi'
fly_ethanol = '/Users/fisherguest/Documents/sansa images/08152025/SC-0815-fly?.czi'     # this one not done yet!
fly3_left = '/Users/fisherguest/Documents/sansa images/08202025/SC-0820-fly3-left.czi'
fly3_right = '/Users/fisherguest/Documents/sansa images/08202025/SC-0820-fly3-right.czi'
fly4 = '/Users/fisherguest/Documents/sansa images/08202025/SC-0820-fly4.czi'


In [13]:
# import image:
img = AICSImage(fly3_left)

# Get the data in (C, Z, Y, X)
# C: Channels (e.g., the different lasers/fluorophores); Z: Z-stacks (slices in depth); Y and X: The spatial dimensions (height and width)
# i don't have S or T.
data = img.get_image_data("CZYX", S=0, T=0)

# checked using "previous TQ method": (633 is the first channel date[0]), 568 is first (no actually second) channel, 488 is second (no actually third) channel.
red_channel   = data[1]
green_channel = data[2]

In [14]:
# open image in napari to draw labels:
viewer = napari.Viewer()

# Add the red channel
viewer.add_image(
    red_channel, 
    name='568 Channel',
    colormap='red',
    #contrast_limits=(0, 4000), # adjust this if needed
    blending='additive'
)
# Add the green channel
viewer.add_image(
    green_channel, 
    name='488 Channel',
    colormap='green',
    #contrast_limits=(50, 800), # adjust this if needed
    blending='additive'
)

napari.run()

In [15]:
# File paths
label_file = '/Users/fisherguest/Documents/sansa images/08202025/roi/fly3-left.tif'


# Load the label mask (the TIFF file you saved from napari)
label_mask = tf.imread(label_file)
# for the label tif file obtained in previous TQ method, need the below precessing:
#squeezed_mask = np.squeeze(label_mask) # so change from shape: (1, 2, Z, Y, X) to (2, Z, Y, X)
#squeezed_mask_green = squeezed_mask[1] # so only extract the green (second) channel.
print("Label mask shape:", label_mask.shape)
print("Green channel shape:", green_channel.shape)

# Check that the dimensions match (e.g., both are 3D)
if green_channel.shape != label_mask.shape:
    raise ValueError("The dimensions of the green channel and the label mask do not match. "
                     "Please verify that your label mask corresponds to the correct z, y, and x dimensions.")

Label mask shape: (104, 1839, 1839)
Green channel shape: (104, 1839, 1839)


In [16]:
# check which z stack I drew the label on.
sums = label_mask.sum(axis=(1,2))
painted_slices = np.where(sums > 0)[0]
print("You painted on slice(s):", painted_slices)

You painted on slice(s): [62]


In [17]:
z_ranges = [
(1, 87),
(27, 98),
(63, 103),
(59, 103),
(39, 103),
(20, 97),
(15, 100),
(0, 89),
(181, 268),
(151, 261),
(128, 254),
(90, 218),
(92, 206),
(107, 208),
(71, 190),
(55, 182),
(42, 139),
(42, 134),
]

In [9]:
# variable input cell
red_threshold_lower = 250
green_threshold_lower = 200

In [18]:
n_glom    = int(label_mask.max())  # should be 18
mean_red   = np.zeros(n_glom, dtype=float)
mean_green = np.zeros(n_glom, dtype=float)
total_red   = np.zeros(n_glom, dtype=float)
total_green = np.zeros(n_glom, dtype=float)
for i in range(1, n_glom+1):
    # 1) find the slice you painted on:
    zs, ys, xs = np.where(label_mask == i)  # THIS COMMENT WILL SOLVE MANY POTENTIAL QUESTIONS: label_mask size is in this format: (Z, Y, X)
    z_draw = zs[0]                          # AICSImage makes the image format in this way! (i don't know why not in X->Y->Z). This makes z_draw the z layer that you draw labels on.
    # 2) extract the 2D ROI on that slice:
    roi2d = (label_mask[z_draw] == i)   # label_mask[z_draw] gives 2d array (Y, X): entries are the integer labels (background:0, ROIs:1-18)
                                        # label_mask[z_draw] == i: give each pixel True of False based on if it's has label 1. So now on this 2D sheet, only the pixels you draw (eg.) label 1 on has a T value.
    # 3) build a 1D mask for your Z‑range:
    z0, z1   = z_ranges[i-1]            # z0=a lowest z layer that the current label reaches, while z1=the highesr z laye the current label goes to.
    Z, Y, X  = label_mask.shape         # label_mask is the full X * full Y * full Z that your current image (eg. fly1) has.
    mask_z   = (np.arange(Z) >= z0) & (np.arange(Z) <= z1)  # np.arange(Z): generates the array [0, 1, 2, …, Z−1]
                                                            # np.arange(Z) >= z0: generates the array [False, F, ...T, T, ... F]
                                                            # () & (): generates the array with True only for indices between z0 and z1.
    # 4) make the final 3d mask:
    roi = mask_z[:, None, None] & roi2d[None, :, :]         # mask_z[:, None, None]: mask_z is originally 1d array, but using None it adds on dimension with length of 1
                                                                # so now it's a 3d array with the first dimension (or first element) length of z-stack range, the second and the third length of 1
                                                            # mask_z[:, None, None] size: (z-stack range, 1, 1)
                                                            # roi2d[None, :, :] size: (1, 1863, 1863) with some x and some y True.
                                                            # Then, the "&" step: for example:
                                                                    # mz goes from (3,1,1) → (3,2,2) by repeating the single 1×1 mask across the 2×2 grid in X and Y.
                                                                    # r2 goes from (1,2,2) → (3,2,2) by repeating the same 2×2 slice across the 3 Z‑layers.
                                                            # in this new 3d array: first dimension (element) is Z!
                                                            # so: in this new 3d array:
                                                                    # first look at mask_z, if it is true, that means there is label in this z-stack,
                                                                        # and thus i will allow the Y and X info from roi2d to get copied into this Z dimension.
                                                                    # if it's False, then there is no label on this z-stack,
                                                                        # so i won't allow roi2d to copy anything to here, and this whole Z dimension will have all False values
            # for example: mask_z = np.array([ True, False,  True ]), roi2d  = np.array([[ True, False], [False,  True]]),
                # then roi = array([[[ True, False], [False,  True]],   # z=0 slice uses roi2d because mask_z[0] is True
                #                   [[False, False], [False, False]],   # z=1 slice is all False because mask_z[1] is False
                #                   [[ True, False], [False,  True]]])  # z=2 slice repeats roi2d because mask_z[2] is True.
    # — b) zero out outside your propagated ROI —
    red_sub   = red_channel[z0:z1+1]   * roi[z0:z1+1]      # red_channel[z0:z1+1]: takes only the z-stacks from the original image (red channel only) that are within the current label z-stack range
                                                           # roi[z0:z1+1]: only slicing in the first (Z) dimension!
                                                           # after multiplication: it's a smaller Z * 1863 * 1863 3d array, with the in-label pixels having value of 1*it's original value (because roi[z0:z1+1] has T or F entries, so multiplying a T=*1 while multiplying a F=*0)
    green_sub = green_channel[z0:z1+1]                     # only select out the correct z stacks, haven't picked the inner pixels yet!
    # — c) threshold & extract means —
    mask      = (red_sub > red_threshold_lower)            # after making the correct label in each z stack, "mask" is the mask for correct pixels.
    red_vals   = red_sub  [mask]                           # only extract the value from the pixel in "mask"
    green_vals = green_sub[mask]
    mean_red[i-1]   = red_vals.mean()   if red_vals.size   else np.nan
    mean_green[i-1] = green_vals.mean() if green_vals.size else np.nan
    total_red[i-1]   = red_vals.sum()   if red_vals.size   else np.nan
    total_green[i-1] = green_vals.sum() if green_vals.size else np.nan


In [19]:
df_means = pd.DataFrame(
    [mean_red, mean_green, total_red, total_green],
    index=['mean_red', 'mean_green', 'total_red', 'total_green']
)
df_means.T

,mean_red,mean_green,total_red,total_green
0,1107.438916,157.804330,690922280.0,98452859.0
1,1040.314452,154.552275,453983864.0,67445222.0
2,1033.271272,134.208243,167313484.0,21731804.0
3,1063.630910,177.015947,328697051.0,54703769.0
4,973.706051,157.401884,308155570.0,49814076.0
5,961.750755,179.697133,233182241.0,43568648.0
6,1128.311185,203.977185,299349984.0,54116779.0
7,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN


In [20]:
df_means = pd.DataFrame(
    [mean_red, mean_green, total_red, total_green],
    index=['mean_red', 'mean_green', 'total_red', 'total_green']
)

df_means.T.to_csv(f"fly3-left.csv")

In [ ]:
# this cell is for when one brain is split into 2 images, and need to have the values combined into one csv...

# build the dataframe
data = {
    "mean_red": [
        1107.43891570977, 1040.31445194791, 1033.27127206255, 1063.63090996754,
        973.706051308626, 961.750754776124, 1128.31118549007, 2213.27164855665,
        1143.13569236239, 1401.68158922517, 1327.22941490938, 1321.87897523149,
        1224.81290322581, 1184.34530380136, 993.846762389452, 2060.9203013386,
        1248.3109130498, 1081.44861004043
    ],
    "mean_green": [
        157.804329916075, 154.552275367732, 134.208243271618, 177.015946517039,
        157.401883865178, 179.697132675619, 203.977185007614, 197.285131692312,
        136.876971025656, 155.999166995057, 170.389243158392, 178.982749751219,
        179.187649604191, 175.596716734685, 156.832377572122, 229.75799142284,
        182.25826681584, 196.988350140805
    ],
    "total_red": [
        690922280.0, 453983864.0, 167313484.0, 328697051.0,
        308155570.0, 233182241.0, 299349984.0, 3825049101.0,
        482317527.0, 718504786.0, 935971474.0, 1253966121.0,
        846333468.0, 643960519.0, 338817269.0, 3169293544.0,
        497629159.0, 137480236.0
    ],
    "total_green": [
        98452859.0, 67445222.0, 21731804.0, 54703769.0,
        49814076.0, 43568648.0, 54116779.0, 340954675.0,
        57751816.0, 79965485.0, 120159687.0, 169787332.0,
        123816874.0, 95476676.0, 53466510.0, 353322988.0,
        72655800.0, 25042341.0
    ]
}

fly3_df = pd.DataFrame(data)

# save it as fly3.csv in current directory
fly3_df.to_csv("fly3.csv", index=True)

In [29]:
# check the mask in napari
propagated_labels = np.zeros_like(label_mask, dtype=np.int32)  # shape (Z, Y, X)
# Your existing loop, with one extra line to “paint” into propagated_labels
for i in range(1, n_glom+1):
    # 1) find the slice you painted on:
    zs, ys, xs = np.where(label_mask == i)
    z_draw = zs[0]
    # 2) 2D ROI on drawn slice
    roi2d = (label_mask[z_draw] == i)
    # 3) Z‑range mask
    z0, z1 = z_ranges[i-1]
    Z, Y, X = label_mask.shape
    mask_z = (np.arange(Z) >= z0) & (np.arange(Z) <= z1)
    # 4) broadcast to 3D
    roi = mask_z[:, None, None] & roi2d[None, :, :]
    # 5) your threshold step and mean extraction
    red_sub   = red_channel[z0:z1+1]   * roi[z0:z1+1]
    mask_thr  = (red_sub > red_threshold_lower)
    # … compute mean_red[i-1], mean_green[i-1] as before …
    # — NEW: paint this ROI into your propagated label map —
    propagated_labels[roi & (red_channel > red_threshold_lower)] = i
# Save the new propagated label volume to TIFF
tf.imwrite('propagated_glom_labels.tif', propagated_labels.astype(np.uint16))
# Now open it in Napari alongside your images
viewer = napari.Viewer()
viewer.add_image(red_channel,   name='568 nm', colormap='red',   blending='additive')
viewer.add_image(green_channel, name='488 nm', colormap='green', blending='additive')
viewer.add_labels(propagated_labels, name='Propagated ROIs', opacity=0.6)
napari.run()



2025-08-17 17:32:03.380 python[99767:111961287] _TIPropertyValueIsValid called with 16 on nil context!
2025-08-17 17:32:03.380 python[99767:111961287] imkxpc_getApplicationProperty:reply: called with incorrect property value 16, bailing.
2025-08-17 17:32:03.380 python[99767:111961287] Text input context does not respond to _valueForTIProperty:


In [ ]:
# don't know why i have this cell...but i will just keep it here...



props_green = regionprops_table(
    label_mask, 
    intensity_image=green_channel,
    properties=('label', 'area', 'mean_intensity', 'max_intensity', 'min_intensity')
)
df_green = pd.DataFrame(props_green)
# Rename columns to indicate green measurements
df_green = df_green.rename(columns={
    'label': 'label',
    'area': 'area',
    'mean_intensity': 'green_mean_intensity',
    'max_intensity': 'green_max_intensity',
    'min_intensity': 'green_min_intensity'
})

props_red = regionprops_table(
    label_mask, 
    intensity_image=red_channel,
    properties=('mean_intensity', 'max_intensity', 'min_intensity')
)
df_red = pd.DataFrame(props_red)
# Rename columns to indicate red measurements
df_red = df_red.rename(columns={
    'mean_intensity': 'red_mean_intensity',
    'max_intensity': 'red_max_intensity',
    'min_intensity': 'red_min_intensity'
})

df_combined = pd.concat([df_green, df_red], axis=1)
df_combined

,label,area,green_mean_intensity,green_max_intensity,green_min_intensity,red_mean_intensity,red_max_intensity,red_min_intensity
0,1,25892.0,26.886258,150.0,3.0,89.018925,1286.0,0.0
1,2,30601.0,34.594000,427.0,4.0,228.817816,7016.0,0.0
2,3,38668.0,46.797352,504.0,7.0,314.064498,6968.0,0.0
3,4,34910.0,41.592237,304.0,5.0,257.410771,7312.0,0.0
4,5,42942.0,50.595547,1168.0,4.0,241.474454,6336.0,0.0
5,6,39097.0,47.902371,402.0,5.0,218.569225,9609.0,0.0
6,7,42310.0,93.945805,960.0,7.0,1647.427393,13304.0,0.0
7,8,47045.0,35.267829,329.0,5.0,123.847316,3310.0,0.0
8,9,36320.0,26.517098,259.0,2.0,55.211151,1109.0,0.0
9,10,34457.0,35.726442,156.0,2.0,82.081261,1029.0,0.0
